# Hindi Subword Tokenizer Ablation Study (BPE vs WordPiece)

**Goal:** Compare Byte-Pair Encoding (BPE) and WordPiece tokenizers on Hindi text, and study how (1) training-data size and (2) vocabulary size affect tokenization efficiency (measured as average tokens per sentence — lower is better compression).

**Pipeline:** Load raw corpus → sentence-split → stratified train/dev/test split → build training subsets of varying size → train BPE & WordPiece tokenizers → evaluate on held-out dev/test sentences.


In [1]:
from datasets import load_dataset

# Load the hindi subset of the dataset
dataset = load_dataset("ai4bharat/IndicCorpV2", "indiccorp_v2", split="hin_Deva",streaming=True)


README.md:   0%|          | 0.00/4.01k [00:00<?, ?B/s]

In [2]:
dataset

IterableDataset({
    features: Unknown,
    num_shards: 3
})

In [3]:
for example in dataset.take(3):
    print(example)
    print("-" * 100)

{'text': 'लोगों को बिलों संबंधी सुविधा देना ही उनका काम'}
----------------------------------------------------------------------------------------------------
{'text': ''}
----------------------------------------------------------------------------------------------------
{'text': 'इनेलो 1987 में उस वक्त ऐसे ही दोराहे पर खड़ी थी, जब पूर्व उपप्रधानमंत्री देवीलाल ने अपने पुत्र ओमप्रकाश चौटाला को अपना राजनीतिक उत्तराधिकारी घोषित किया था। हालांकि तब पार्टी पर देवीलाल की मजबूत पकड़ के चलते पार्टी टूटने से बच गई थी। 1989 में देवीलाल केन्द्र की राजनीति में सक्रिय हो गए थे और उनके उपप्रधानमंत्री बनने के पश्चात् उनके तीन बेटों जगदीश सिंह, रणजीत सिंह और ओमप्रकाश चौटाला में से रणजीत और ओमप्रकाश के बीच हरियाणा में उनकी राजनीतिक विरासत को लेकर जंग शुरू हो गई थी। उन परिस्थितियों में देवीलाल ने कड़ा निर्णय लेते हुए पार्टी की बागडोर ओमप्रकाश चौटाला के हवाले कर दी थी, जिसके बाद रणजीत की बगावत का असर पार्टी, संगठन और उनकी सरकार पर भी पड़ा था। उस समय रणजीत की नाराजगी के चलते उनके समर्थन में कई कैबिनेट मं

In [4]:
import re

def sentence_tokenize_hindi(paragraph):

    if not isinstance(paragraph, str):
        return []

    paragraph = paragraph.strip()

    # Split after Hindi full stop, question mark,
    # or exclamation mark
    sentences = re.split(
        r'(?<=[।?!])\s+',
        paragraph
    )

    # Remove empty and extremely short sentences
    sentences = [
        sentence.strip()
        for sentence in sentences
        if len(sentence.strip()) > 1
    ]

    return sentences

In [5]:
paragraph = """
भारत एक विशाल देश है। इसकी कई भाषाएँ और संस्कृतियाँ हैं।
दिल्ली भारत की राजधानी है!
क्या आप भारत गए हैं?
"""

print(sentence_tokenize_hindi(paragraph))

['भारत एक विशाल देश है।', 'इसकी कई भाषाएँ और संस्कृतियाँ हैं।', 'दिल्ली भारत की राजधानी है!', 'क्या आप भारत गए हैं?']


In [7]:
from tqdm import tqdm

TARGET_SENTENCES = 1_050_000

sentences = []

for example in tqdm(dataset):

    paragraph = example["text"]

    paragraph_sentences = sentence_tokenize_hindi(
        paragraph
    )

    for sentence in paragraph_sentences:

        # Remove extremely short sentences
        if len(sentence.split()) >= 3:
            sentences.append(sentence)

    if len(sentences) >= TARGET_SENTENCES:
        break

752108it [00:19, 37629.45it/s]


In [8]:
print("Total sentences collected:", len(sentences))

Total sentences collected: 1050000


In [9]:
import pandas as pd

sentences_df = pd.DataFrame({
    "sentence": sentences
})

sentences_df.to_csv(
    "hindi_sentences.csv",
    index=False,
    encoding="utf-8"
)

In [10]:
sentences_df["length"] = (
    sentences_df["sentence"]
    .apply(lambda x: len(x.split()))
)

print(
    sentences_df["length"].describe()
)

count    1.050000e+06
mean     2.061190e+01
std      2.155925e+01
min      3.000000e+00
25%      1.100000e+01
50%      1.600000e+01
75%      2.400000e+01
max      3.276000e+03
Name: length, dtype: float64


In [11]:
sentences[0]

'लोगों को बिलों संबंधी सुविधा देना ही उनका काम'

## 2. Sentence Length Analysis & Stratified Train/Dev/Test Split

We bucket sentences by word length and sample dev/test sets so that every length range (short, medium, long sentences) is represented proportionally — a plain random split could accidentally put mostly short or mostly long sentences into dev/test, biasing the token-count metrics.


In [12]:
def get_length_bucket(length):

    if length <= 5:
        return "1-5"

    elif length <= 10:
        return "6-10"

    elif length <= 20:
        return "11-20"

    elif length <= 30:
        return "21-30"

    elif length <= 50:
        return "31-50"

    else:
        return "50+"

In [13]:
sentences_df["length_bucket"] = (
    sentences_df["length"]
    .apply(get_length_bucket)
)

In [14]:
print(
    sentences_df["length_bucket"]
    .value_counts()
)

length_bucket
11-20    457031
21-30    208361
6-10     198278
31-50    102784
50+       45832
1-5       37714
Name: count, dtype: int64


In [15]:
import numpy as np
import pandas as pd


def stratified_length_split(
    df,
    dev_size=1000,
    test_size=1000,
    random_state=42
):

    dev_parts = []
    test_parts = []

    # Distribution of sentence length buckets
    bucket_counts = (
        df["length_bucket"]
        .value_counts()
    )

    total_samples = len(df)

    for i, (bucket, count) in enumerate(
        bucket_counts.items()
    ):

        # Get all sentences belonging to this bucket
        bucket_df = df[
            df["length_bucket"] == bucket
        ].sample(
            frac=1,
            random_state=random_state + i
        )

        # Number of development samples
        dev_n = round(
            (count / total_samples) *
            dev_size
        )

        # Number of test samples
        test_n = round(
            (count / total_samples) *
            test_size
        )

        # Development samples
        dev_part = bucket_df.iloc[
            :dev_n
        ]

        # Test samples
        test_part = bucket_df.iloc[
            dev_n:dev_n + test_n
        ]

        dev_parts.append(dev_part)

        test_parts.append(test_part)


    # Combine all buckets
    dev_df = pd.concat(
        dev_parts
    )

    test_df = pd.concat(
        test_parts
    )

    # If rounding causes slightly more/fewer than 1000,
    # adjust to exactly 1000
    dev_df = dev_df.sample(
        n=dev_size,
        random_state=random_state
    )

    test_df = test_df.sample(
        n=test_size,
        random_state=random_state
    )

    # Training data = everything except dev and test
    train_df = df.drop(
        dev_df.index.union(test_df.index)
    )

    return train_df, dev_df, test_df

In [16]:
train_df, dev_df, test_df = stratified_length_split(
    sentences_df,
    dev_size=1000,
    test_size=1000,
    random_state=42
)

In [17]:
print("Training samples   :", len(train_df))
print("Development samples:", len(dev_df))
print("Test samples       :", len(test_df))

Training samples   : 1048000
Development samples: 1000
Test samples       : 1000


In [18]:
distribution_comparison = pd.DataFrame({

    "Original (%)":
    sentences_df["length_bucket"]
    .value_counts(normalize=True)
    .sort_index()
    * 100,

    "Development (%)":
    dev_df["length_bucket"]
    .value_counts(normalize=True)
    .sort_index()
    * 100,

    "Test (%)":
    test_df["length_bucket"]
    .value_counts(normalize=True)
    .sort_index()
    * 100

}).fillna(0)

print(
    distribution_comparison.round(2)
)

               Original (%)  Development (%)  Test (%)
length_bucket                                         
1-5                    3.59              3.6       3.6
11-20                 43.53             43.5      43.5
21-30                 19.84             19.8      19.8
31-50                  9.79              9.8       9.8
50+                    4.36              4.4       4.4
6-10                  18.88             18.9      18.9


In [22]:
train_df = train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(train_df.head())

                                            sentence  length length_bucket
0  जिले के सभी सरकारी अस्पतालों में ओपीडी को अगले...      17         11-20
1  सिंधिया के इस्तीफा देते ही सामने आया हरियाणा क...      13         11-20
2  काफी देर तक कांग्रेसियों और ईसी सदस्यों के बीच...      12         11-20
3  मोबाइल नंबर पोर्टिबिलिटी (एमएनपी) के संशोधित न...      17         11-20
4  वहीं, प्रधानमंत्री नरेंद्र मोदी ने राहुल गांधी...      13         11-20


## 3. Creating Training Subsets of Different Sizes

We carve four nested subsets (100K / 300K / 500K / 1M sentences) from the same training pool. This is the independent variable for **Experiment 1**.


In [23]:
TRAIN_SIZES = [
    100_000,
    300_000,
    500_000,
    1_000_000
]

In [24]:
train_subsets = {}

for size in TRAIN_SIZES:

    if len(train_df) >= size:

        train_subsets[size] = train_df.iloc[:size].copy()

        print(
            f"{size:,} samples created"
        )

    else:

        print(
            f"Not enough training samples for {size:,}"
        )

100,000 samples created
300,000 samples created
500,000 samples created
1,000,000 samples created


In [25]:
VOCAB_SIZES = [
    20_000,
    30_000,
    50_000
]

In [26]:
train_text_100k = train_subsets[100_000]["sentence"].tolist()

print(len(train_text_100k))
print(train_text_100k[0])

100000
जिले के सभी सरकारी अस्पतालों में ओपीडी को अगले आदेश तक बंद रखने का निर्देश दिया गया।


## 4. Manual BPE Implementation (Educational Demo Only)

This hand-written version shows *how BPE actually works* internally: represent each word as characters, count adjacent symbol-pair frequencies, and greedily merge the most frequent pair, repeating until the target vocabulary size is reached.

> **Important for viva:** this manual implementation is **not used** for the actual experiments below — it's kept purely to explain the algorithm. All real tokenizer training uses the HuggingFace `tokenizers` library (Section 5) because it is fast (Rust-backed) and battle-tested.


In [27]:
from collections import Counter

def get_word_frequencies(texts):

    word_freq = Counter()

    for sentence in texts:
        words = sentence.split()

        for word in words:
            word_freq[word] += 1

    return word_freq

In [28]:
def initialize_bpe_vocab(word_freq):

    vocab = {}

    for word, freq in word_freq.items():

        tokens = tuple(list(word) + ["</w>"])

        vocab[tokens] = freq

    return vocab

def get_pair_counts(vocab):

    pairs = Counter()

    for tokens, freq in vocab.items():

        for i in range(len(tokens) - 1):

            pair = (
                tokens[i],
                tokens[i + 1]
            )

            pairs[pair] += freq

    return pairs

def merge_pair(pair, vocab):

    new_vocab = {}

    merged_token = "".join(pair)

    for tokens, freq in vocab.items():

        new_tokens = []
        i = 0

        while i < len(tokens):

            if (
                i < len(tokens) - 1
                and tokens[i] == pair[0]
                and tokens[i + 1] == pair[1]
            ):

                new_tokens.append(merged_token)
                i += 2

            else:

                new_tokens.append(tokens[i])
                i += 1

        new_vocab[tuple(new_tokens)] = freq

    return new_vocab



### Manual WordPiece Implementation (Educational Demo Only)

BPE (above) always merges the pair with the **highest raw frequency**. WordPiece instead scores every candidate pair with a **likelihood score** and merges the pair with the **highest score**:

$$\text{score}(a, b) = \dfrac{\text{freq}(a, b)}{\text{freq}(a) \times \text{freq}(b)}$$

This favors a pair that co-occurs often *relative to* how often its two parts occur individually — so two rare symbols that almost always appear together can outscore a very frequent pair whose parts are themselves extremely common (and thus already 'expected' to co-occur by chance).

> Same caveat as the manual BPE demo: this is for understanding the mechanism only. The real WordPiece tokenizers used in the experiments are trained with `models.WordPiece` + `WordPieceTrainer` from the HuggingFace `tokenizers` library (Section 5), which implements this same scoring rule internally but far more efficiently.


In [29]:
def get_pair_scores(vocab):
    """
    WordPiece merge scoring:
        score(a, b) = freq(a, b) / (freq(a) * freq(b))
    Reuses the same `vocab` structure as the BPE demo above:
    {tuple_of_symbols: frequency}.
    """
    pair_counts = Counter()
    symbol_counts = Counter()

    for tokens, freq in vocab.items():
        for symbol in tokens:
            symbol_counts[symbol] += freq
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pair_counts[pair] += freq

    scores = {}
    for pair, count in pair_counts.items():
        left_freq = symbol_counts[pair[0]]
        right_freq = symbol_counts[pair[1]]
        scores[pair] = count / (left_freq * right_freq)

    return scores


def manual_train_wordpiece_demo(word_freq, num_merges=10):
    """
    Runs a few WordPiece merge steps by hand and prints, at each step,
    the chosen pair, its score, and the resulting vocabulary size —
    to make the scoring rule concrete.
    """
    vocab = initialize_bpe_vocab(word_freq)  # same char-level init as BPE

    for step in range(num_merges):
        scores = get_pair_scores(vocab)

        if not scores:
            break

        best_pair = max(scores, key=scores.get)
        best_score = scores[best_pair]

        vocab = merge_pair(best_pair, vocab)

        print(f"Step {step + 1}: merged {best_pair} -> "
              f"'{''.join(best_pair)}'  (score = {best_score:.6f})")

    return vocab

In [30]:
# Small-scale demo so the merge steps are actually readable
demo_word_freq = get_word_frequencies(train_text_100k[:2000])

print("BPE merge (by raw frequency) vs WordPiece merge (by likelihood score) "
      "on the same starting vocabulary:\n")

print("--- WordPiece merges ---")
_ = manual_train_wordpiece_demo(demo_word_freq, num_merges=10)

BPE merge (by raw frequency) vs WordPiece merge (by likelihood score) on the same starting vocabulary:

--- WordPiece merges ---
Step 1: merged ('९', '७') -> '९७'  (score = 0.500000)
Step 2: merged ('२', '०') -> '२०'  (score = 0.250000)
Step 3: merged ('३', '०') -> '३०'  (score = 0.500000)
Step 4: merged ('२०', '०') -> '२००'  (score = 0.500000)
Step 5: merged ('२००', '८') -> '२००८'  (score = 0.500000)
Step 6: merged ('}', '}') -> '}}'  (score = 0.250000)
Step 7: merged ('२०', '१') -> '२०१'  (score = 0.200000)
Step 8: merged ('२०१', '७') -> '२०१७'  (score = 1.000000)
Step 9: merged ('१', '९७') -> '१९७'  (score = 0.250000)
Step 10: merged ('१९७', '१') -> '१९७१'  (score = 0.333333)


## 5. Production Tokenizer Training (HuggingFace `tokenizers` library)

`train_bpe()` trains a real BPE model; `train_wordpiece()` trains a WordPiece model. Both use the same `Whitespace` pre-tokenizer so the *only* difference between the two is the merge/scoring algorithm:
- **BPE** merges the pair with the highest raw frequency.
- **WordPiece** merges the pair that maximizes `freq(pair) / (freq(token1) * freq(token2))` (a likelihood-style score), which favors merges that are frequent *relative to* their parts, not just frequent in absolute terms.


In [31]:
!pip install -q tokenizers

In [32]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

In [33]:
def train_bpe(texts, vocab_size):

    tokenizer = Tokenizer(
        models.BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = (
        pre_tokenizers.Whitespace()
    )

    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train_from_iterator(
        texts,
        trainer=trainer
    )

    return tokenizer

In [34]:
bpe_100k_20k = train_bpe(
    train_subsets[100_000]["sentence"].tolist(),
    vocab_size=20_000
)

In [35]:
print(
    "Vocabulary size:",
    bpe_100k_20k.get_vocab_size()
)

Vocabulary size: 20000


In [36]:
sample = dev_df["sentence"].iloc[0]

tokens = bpe_100k_20k.encode(sample).tokens

print("Sentence:")
print(sample)

print("\nBPE tokens:")
print(tokens)

print("\nNumber of tokens:")
print(len(tokens))

Sentence:
बुजुर्गों के लिए टैक्सी सेवा उपलब्ध है।

BPE tokens:
['बुजुर्गों', 'के', 'लिए', 'टैक्सी', 'सेवा', 'उपलब्ध', 'है', '।']

Number of tokens:
8


In [37]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers


def train_wordpiece(texts, vocab_size):
    """
    Train a WordPiece tokenizer.

    Parameters:
        texts: list of training sentences
        vocab_size: desired vocabulary size

    Returns:
        trained WordPiece tokenizer
    """

    tokenizer = Tokenizer(
        models.WordPiece(
            unk_token="[UNK]"
        )
    )

    # Split input into words before applying WordPiece
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

    trainer = trainers.WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train_from_iterator(
        texts,
        trainer=trainer
    )

    return tokenizer

In [38]:
wp_tokenizer = train_wordpiece(
    train_subsets[100_000]["sentence"].tolist(),
    20_000
)

print(
    "WordPiece vocabulary size:",
    wp_tokenizer.get_vocab_size()
)

WordPiece vocabulary size: 20000


In [39]:
sample_sentence = dev_df["sentence"].iloc[0]

print("Original sentence:")
print(sample_sentence)

print("\nWordPiece tokens:")

tokens = wp_tokenizer.encode(
    sample_sentence
).tokens

print(tokens)
print("\nNumber of tokens:", len(tokens))

Original sentence:
बुजुर्गों के लिए टैक्सी सेवा उपलब्ध है।

WordPiece tokens:
['बुजुर्गों', 'के', 'लिए', 'टैक्सी', 'सेवा', 'उपलब्ध', 'है', '।']

Number of tokens: 8


## 6. Experiment 1 — Effect of TRAINING DATA SIZE on Tokenization
**Controlled variable:** vocabulary size fixed at 20,000 for both tokenizers.
**Independent variable:** training-sentence count (100K → 1M).
**What to watch for:** does giving the tokenizer more sentences to learn merges from change how efficiently it compresses unseen (dev-set) text?


In [40]:
bpe_tokenizers = {}

for train_size in TRAIN_SIZES:

    print(
        f"\nTraining BPE on {train_size:,} sentences..."
    )

    texts = train_subsets[
        train_size
    ]["sentence"].tolist()

    bpe_tokenizers[train_size] = train_bpe(
        texts,
        vocab_size=20_000
    )

    print(
        "Vocabulary size:",
        bpe_tokenizers[
            train_size
        ].get_vocab_size()
    )


Training BPE on 100,000 sentences...
Vocabulary size: 20000

Training BPE on 300,000 sentences...
Vocabulary size: 20000

Training BPE on 500,000 sentences...
Vocabulary size: 20000

Training BPE on 1,000,000 sentences...
Vocabulary size: 20000


In [41]:
sample_sentence = dev_df["sentence"].iloc[0]

for train_size in TRAIN_SIZES:

    tokenizer = bpe_tokenizers[train_size]

    tokens = tokenizer.encode(
        sample_sentence
    ).tokens

    print("\n" + "=" * 70)

    print(
        f"BPE - Training Size: {train_size:,}"
    )

    print("Tokens:")
    print(tokens)

    print(
        "Number of tokens:",
        len(tokens)
    )


BPE - Training Size: 100,000
Tokens:
['बुजुर्गों', 'के', 'लिए', 'टैक्सी', 'सेवा', 'उपलब्ध', 'है', '।']
Number of tokens: 8

BPE - Training Size: 300,000
Tokens:
['बुजुर्गों', 'के', 'लिए', 'टैक्सी', 'सेवा', 'उपलब्ध', 'है', '।']
Number of tokens: 8

BPE - Training Size: 500,000
Tokens:
['बुजुर्गों', 'के', 'लिए', 'टैक्सी', 'सेवा', 'उपलब्ध', 'है', '।']
Number of tokens: 8

BPE - Training Size: 1,000,000
Tokens:
['बुजुर्गों', 'के', 'लिए', 'टैक्सी', 'सेवा', 'उपलब्ध', 'है', '।']
Number of tokens: 8


In [42]:
def find_differentiating_sentence(tokenizer_dict, keys, sentences, min_variants=2):
    """
    Scan `sentences` for the first one whose token COUNT differs across the tokenizers
    in tokenizer_dict[key] for key in keys. Returns (sentence, {key: tokens}, num_sentences_that_differ).
    """
    best = None
    num_differing = 0

    for sentence in sentences:
        token_lists = {
            key: tokenizer_dict[key].encode(sentence).tokens
            for key in keys
        }
        lengths = {len(t) for t in token_lists.values()}

        if len(lengths) >= min_variants:
            num_differing += 1
            if best is None:
                best = (sentence, token_lists)

    return best, num_differing


In [43]:
search_pool = dev_df["sentence"].tolist()

best, num_differing = find_differentiating_sentence(
    bpe_tokenizers, TRAIN_SIZES, search_pool
)

print(f"{num_differing} / {len(search_pool)} dev sentences tokenize differently "
      f"across training sizes for BPE.\n")

if best:
    sentence, token_lists = best
    print("Example sentence:", sentence, "\n")
    for train_size in TRAIN_SIZES:
        tokens = token_lists[train_size]
        print(f"BPE - Training Size: {train_size:,}")
        print("Tokens:", tokens)
        print("Number of tokens:", len(tokens), "\n")
else:
    print("No differentiating sentence found in the dev set for BPE across these training sizes.")

224 / 1000 dev sentences tokenize differently across training sizes for BPE.

Example sentence: ‘सेटिंग स्टैंडर्ड की अपनी परंपरागत उद्देश्य की बखूबी पूर्ति करते हुए देश के लिए होनहार पायलटों की पीढ़ी तैयार करने के लिए उन्होंने बीएफटीएस की भी प्रशंसा की। 

BPE - Training Size: 100,000
Tokens: ['‘', 'सेटिंग', 'स्टैंडर्ड', 'की', 'अपनी', 'परंपरागत', 'उद्देश्य', 'की', 'बखूबी', 'पूर्ति', 'करते', 'हुए', 'देश', 'के', 'लिए', 'होन', 'हार', 'पायल', 'टों', 'की', 'पीढ़ी', 'तैयार', 'करने', 'के', 'लिए', 'उन्होंने', 'बी', 'एफ', 'टीएस', 'की', 'भी', 'प्रशंसा', 'की', '।']
Number of tokens: 34 

BPE - Training Size: 300,000
Tokens: ['‘', 'सेटिंग', 'स्टैंडर्ड', 'की', 'अपनी', 'परंपरागत', 'उद्देश्य', 'की', 'बखूबी', 'पूर्ति', 'करते', 'हुए', 'देश', 'के', 'लिए', 'होनहार', 'पायल', 'टों', 'की', 'पीढ़ी', 'तैयार', 'करने', 'के', 'लिए', 'उन्होंने', 'बी', 'एफ', 'टीएस', 'की', 'भी', 'प्रशंसा', 'की', '।']
Number of tokens: 33 

BPE - Training Size: 500,000
Tokens: ['‘', 'सेटिंग', 'स्टैंडर्ड', 'की', 'अपनी', 'परंपरागत', 'उद्द

In [44]:
wordpiece_tokenizers = {}

for train_size in TRAIN_SIZES:

    print(
        f"\nTraining WordPiece on "
        f"{train_size:,} sentences..."
    )

    texts = train_subsets[
        train_size
    ]["sentence"].tolist()

    wordpiece_tokenizers[train_size] = train_wordpiece(
        texts,
        vocab_size=20_000
    )

    print(
        "Vocabulary size:",
        wordpiece_tokenizers[
            train_size
        ].get_vocab_size()
    )


Training WordPiece on 100,000 sentences...
Vocabulary size: 20000

Training WordPiece on 300,000 sentences...
Vocabulary size: 20000

Training WordPiece on 500,000 sentences...
Vocabulary size: 20000

Training WordPiece on 1,000,000 sentences...
Vocabulary size: 20000


In [45]:
sample_sentence = dev_df["sentence"].iloc[0]

for train_size in TRAIN_SIZES:

    tokenizer = wordpiece_tokenizers[train_size]

    tokens = tokenizer.encode(
        sample_sentence
    ).tokens

    print("\n" + "=" * 70)

    print(
        f"WordPiece - Training Size: "
        f"{train_size:,}"
    )

    print("Tokens:")
    print(tokens)

    print(
        "Number of tokens:",
        len(tokens)
    )


WordPiece - Training Size: 100,000
Tokens:
['बुजुर्गों', 'के', 'लिए', 'टैक्सी', 'सेवा', 'उपलब्ध', 'है', '।']
Number of tokens: 8

WordPiece - Training Size: 300,000
Tokens:
['बुजुर्गों', 'के', 'लिए', 'टैक्सी', 'सेवा', 'उपलब्ध', 'है', '।']
Number of tokens: 8

WordPiece - Training Size: 500,000
Tokens:
['बुजुर्गों', 'के', 'लिए', 'टैक्सी', 'सेवा', 'उपलब्ध', 'है', '।']
Number of tokens: 8

WordPiece - Training Size: 1,000,000
Tokens:
['बुजुर्गों', 'के', 'लिए', 'टैक्सी', 'सेवा', 'उपलब्ध', 'है', '।']
Number of tokens: 8


In [46]:
best, num_differing = find_differentiating_sentence(
    wordpiece_tokenizers, TRAIN_SIZES, search_pool
)

print(f"{num_differing} / {len(search_pool)} dev sentences tokenize differently "
      f"across training sizes for WordPiece.\n")

if best:
    sentence, token_lists = best
    print("Example sentence:", sentence, "\n")
    for train_size in TRAIN_SIZES:
        tokens = token_lists[train_size]
        print(f"WordPiece - Training Size: {train_size:,}")
        print("Tokens:", tokens)
        print("Number of tokens:", len(tokens), "\n")
else:
    print("No differentiating sentence found in the dev set for WordPiece across these training sizes.")

251 / 1000 dev sentences tokenize differently across training sizes for WordPiece.

Example sentence: इसके तहत पार्टी के राज्य से लेकर पंचायत तक के पदाधिकारी अपने-अपने क्षेत्र में नुक्कड़ बैठक, नुक्कड़ सभा, किसान चौपाल और पदयात्रा करेंगे। 

WordPiece - Training Size: 100,000
Tokens: ['इसके', 'तहत', 'पार्टी', 'के', 'राज्य', 'से', 'लेकर', 'पंचायत', 'तक', 'के', 'पदाधिकारी', 'अपने', '-', 'अपने', 'क्षेत्र', 'में', 'नुक', '##्', '##कड़', 'बैठक', ',', 'नुक', '##्', '##कड़', 'सभा', ',', 'किसान', 'चौपाल', 'और', 'पद', '##यात्रा', 'करेंगे', '।']
Number of tokens: 33 

WordPiece - Training Size: 300,000
Tokens: ['इसके', 'तहत', 'पार्टी', 'के', 'राज्य', 'से', 'लेकर', 'पंचायत', 'तक', 'के', 'पदाधिकारी', 'अपने', '-', 'अपने', 'क्षेत्र', 'में', 'नुक', '##्', '##कड़', 'बैठक', ',', 'नुक', '##्', '##कड़', 'सभा', ',', 'किसान', 'चौपाल', 'और', 'पदयात्रा', 'करेंगे', '।']
Number of tokens: 32 

WordPiece - Training Size: 500,000
Tokens: ['इसके', 'तहत', 'पार्टी', 'के', 'राज्य', 'से', 'लेकर', 'पंचायत', 'तक', 'के',

In [47]:
def get_token_statistics(tokenizer, sentences):

    token_counts = []

    for sentence in sentences:

        tokens = tokenizer.encode(
            sentence
        ).tokens

        token_counts.append(
            len(tokens)
        )

    return {
        "Total Sentences": len(sentences),
        "Total Tokens": sum(token_counts),
        "Average Tokens": sum(token_counts) / len(token_counts),
        "Minimum Tokens": min(token_counts),
        "Maximum Tokens": max(token_counts)
    }

In [48]:
bpe_training_results = []

for train_size in TRAIN_SIZES:

    stats = get_token_statistics(
        bpe_tokenizers[train_size],
        dev_df["sentence"].tolist()
    )

    stats["Tokenizer"] = "BPE"
    stats["Training Size"] = train_size

    bpe_training_results.append(stats)


bpe_training_results_df = pd.DataFrame(
    bpe_training_results
)

bpe_training_results_df = bpe_training_results_df[
    [
        "Tokenizer",
        "Training Size",
        "Total Sentences",
        "Total Tokens",
        "Average Tokens",
        "Minimum Tokens",
        "Maximum Tokens"
    ]
]

bpe_training_results_df

,Tokenizer,Training Size,Total Sentences,Total Tokens,Average Tokens,Minimum Tokens,Maximum Tokens
0,BPE,100000,1000,24450,24.450,4,363
1,BPE,300000,1000,24399,24.399,4,365
2,BPE,500000,1000,24405,24.405,4,365
3,BPE,1000000,1000,24428,24.428,4,366


In [49]:
wordpiece_training_results = []

for train_size in TRAIN_SIZES:

    stats = get_token_statistics(
        wordpiece_tokenizers[train_size],
        dev_df["sentence"].tolist()
    )

    stats["Tokenizer"] = "WordPiece"
    stats["Training Size"] = train_size

    wordpiece_training_results.append(stats)


wordpiece_training_results_df = pd.DataFrame(
    wordpiece_training_results
)

wordpiece_training_results_df = wordpiece_training_results_df[
    [
        "Tokenizer",
        "Training Size",
        "Total Sentences",
        "Total Tokens",
        "Average Tokens",
        "Minimum Tokens",
        "Maximum Tokens"
    ]
]

wordpiece_training_results_df

,Tokenizer,Training Size,Total Sentences,Total Tokens,Average Tokens,Minimum Tokens,Maximum Tokens
0,WordPiece,100000,1000,24744,24.744,4,367
1,WordPiece,300000,1000,24739,24.739,4,371
2,WordPiece,500000,1000,24761,24.761,4,372
3,WordPiece,1000000,1000,24792,24.792,4,374


In [50]:
training_size_comparison = pd.concat(
    [
        bpe_training_results_df,
        wordpiece_training_results_df
    ],
    ignore_index=True
)

training_size_comparison

,Tokenizer,Training Size,Total Sentences,Total Tokens,Average Tokens,Minimum Tokens,Maximum Tokens
0,BPE,100000,1000,24450,24.450,4,363
1,BPE,300000,1000,24399,24.399,4,365
2,BPE,500000,1000,24405,24.405,4,365
3,BPE,1000000,1000,24428,24.428,4,366
4,WordPiece,100000,1000,24744,24.744,4,367
5,WordPiece,300000,1000,24739,24.739,4,371
6,WordPiece,500000,1000,24761,24.761,4,372
7,WordPiece,1000000,1000,24792,24.792,4,374


In [51]:
# --- Side-by-side pivot for Experiment 1: rows = Training Size, columns = Tokenizer ---
pivot_train_size = training_size_comparison.pivot(
    index="Training Size", columns="Tokenizer", values="Average Tokens"
)
pivot_train_size["BPE - WordPiece (diff)"] = (
    pivot_train_size["BPE"] - pivot_train_size["WordPiece"]
)
print("Average tokens per sentence, BPE vs WordPiece, by training size:")
pivot_train_size.round(3)

Average tokens per sentence, BPE vs WordPiece, by training size:


Tokenizer,BPE,WordPiece,BPE - WordPiece (diff)
Training Size,,,
100000,24.450,24.744,-0.294
300000,24.399,24.739,-0.340
500000,24.405,24.761,-0.356
1000000,24.428,24.792,-0.364


## 7. Experiment 2 — Effect of VOCABULARY SIZE on Tokenization
**Controlled variable:** training data fixed at 100,000 sentences for both tokenizers.
**Independent variable:** vocabulary size (20K / 30K / 50K).
**What to watch for:** a larger vocabulary lets the tokenizer store longer/more specific subword units, which should reduce the number of tokens needed per sentence — at the cost of a bigger embedding table in any model built on top of this tokenizer.


In [52]:
fixed_training_data = train_subsets[
    100_000
]["sentence"].tolist()

In [53]:
bpe_vocab_tokenizers = {}

for vocab_size in [20_000, 30_000, 50_000]:

    print(
        f"\nTraining BPE with vocabulary "
        f"size = {vocab_size:,}"
    )

    bpe_vocab_tokenizers[vocab_size] = train_bpe(
        fixed_training_data,
        vocab_size=vocab_size
    )

    print(
        "Actual vocabulary size:",
        bpe_vocab_tokenizers[
            vocab_size
        ].get_vocab_size()
    )


Training BPE with vocabulary size = 20,000
Actual vocabulary size: 20000

Training BPE with vocabulary size = 30,000
Actual vocabulary size: 30000

Training BPE with vocabulary size = 50,000
Actual vocabulary size: 50000


In [54]:
wordpiece_vocab_tokenizers = {}

for vocab_size in [20_000, 30_000, 50_000]:

    print(
        f"\nTraining WordPiece with vocabulary "
        f"size = {vocab_size:,}"
    )

    wordpiece_vocab_tokenizers[vocab_size] = train_wordpiece(
        fixed_training_data,
        vocab_size=vocab_size
    )

    print(
        "Actual vocabulary size:",
        wordpiece_vocab_tokenizers[
            vocab_size
        ].get_vocab_size()
    )


Training WordPiece with vocabulary size = 20,000
Actual vocabulary size: 20000

Training WordPiece with vocabulary size = 30,000
Actual vocabulary size: 30000

Training WordPiece with vocabulary size = 50,000
Actual vocabulary size: 50000


In [55]:
VOCAB_SIZES_USED = [20_000, 30_000, 50_000]
search_pool = dev_df["sentence"].tolist()

best, num_differing = find_differentiating_sentence(
    bpe_vocab_tokenizers, VOCAB_SIZES_USED, search_pool
)

print(f"{num_differing} / {len(search_pool)} dev sentences tokenize differently "
      f"across vocabulary sizes for BPE.\n")

if best:
    sentence, token_lists = best
    print("Example sentence:", sentence, "\n")
    for vocab_size in VOCAB_SIZES_USED:
        tokens = token_lists[vocab_size]
        print(f"BPE - Vocabulary Size: {vocab_size:,}")
        print("Tokens:", tokens)
        print("Number of tokens:", len(tokens), "\n")
else:
    print("No differentiating sentence found in the dev set for BPE across these vocab sizes.")

452 / 1000 dev sentences tokenize differently across vocabulary sizes for BPE.

Example sentence: इसके तहत पार्टी के राज्य से लेकर पंचायत तक के पदाधिकारी अपने-अपने क्षेत्र में नुक्कड़ बैठक, नुक्कड़ सभा, किसान चौपाल और पदयात्रा करेंगे। 

BPE - Vocabulary Size: 20,000
Tokens: ['इसके', 'तहत', 'पार्टी', 'के', 'राज्य', 'से', 'लेकर', 'पंचायत', 'तक', 'के', 'पदाधिकारी', 'अपने', '-', 'अपने', 'क्षेत्र', 'में', 'नु', 'क्कड़', 'बैठक', ',', 'नु', 'क्कड़', 'सभा', ',', 'किसान', 'चौपाल', 'और', 'पदयात्रा', 'करेंगे', '।']
Number of tokens: 30 

BPE - Vocabulary Size: 30,000
Tokens: ['इसके', 'तहत', 'पार्टी', 'के', 'राज्य', 'से', 'लेकर', 'पंचायत', 'तक', 'के', 'पदाधिकारी', 'अपने', '-', 'अपने', 'क्षेत्र', 'में', 'नुक्कड़', 'बैठक', ',', 'नुक्कड़', 'सभा', ',', 'किसान', 'चौपाल', 'और', 'पदयात्रा', 'करेंगे', '।']
Number of tokens: 28 

BPE - Vocabulary Size: 50,000
Tokens: ['इसके', 'तहत', 'पार्टी', 'के', 'राज्य', 'से', 'लेकर', 'पंचायत', 'तक', 'के', 'पदाधिकारी', 'अपने', '-', 'अपने', 'क्षेत्र', 'में', 'नुक्कड़', '

In [56]:
best, num_differing = find_differentiating_sentence(
    wordpiece_vocab_tokenizers, VOCAB_SIZES_USED, search_pool
)

print(f"{num_differing} / {len(search_pool)} dev sentences tokenize differently "
      f"across vocabulary sizes for WordPiece.\n")

if best:
    sentence, token_lists = best
    print("Example sentence:", sentence, "\n")
    for vocab_size in VOCAB_SIZES_USED:
        tokens = token_lists[vocab_size]
        print(f"WordPiece - Vocabulary Size: {vocab_size:,}")
        print("Tokens:", tokens)
        print("Number of tokens:", len(tokens), "\n")
else:
    print("No differentiating sentence found in the dev set for WordPiece across these vocab sizes.")

497 / 1000 dev sentences tokenize differently across vocabulary sizes for WordPiece.

Example sentence: इसके तहत पार्टी के राज्य से लेकर पंचायत तक के पदाधिकारी अपने-अपने क्षेत्र में नुक्कड़ बैठक, नुक्कड़ सभा, किसान चौपाल और पदयात्रा करेंगे। 

WordPiece - Vocabulary Size: 20,000
Tokens: ['इसके', 'तहत', 'पार्टी', 'के', 'राज्य', 'से', 'लेकर', 'पंचायत', 'तक', 'के', 'पदाधिकारी', 'अपने', '-', 'अपने', 'क्षेत्र', 'में', 'नुक', '##्', '##कड़', 'बैठक', ',', 'नुक', '##्', '##कड़', 'सभा', ',', 'किसान', 'चौपाल', 'और', 'पद', '##यात्रा', 'करेंगे', '।']
Number of tokens: 33 

WordPiece - Vocabulary Size: 30,000
Tokens: ['इसके', 'तहत', 'पार्टी', 'के', 'राज्य', 'से', 'लेकर', 'पंचायत', 'तक', 'के', 'पदाधिकारी', 'अपने', '-', 'अपने', 'क्षेत्र', 'में', 'नुक्कड़', 'बैठक', ',', 'नुक्कड़', 'सभा', ',', 'किसान', 'चौपाल', 'और', 'पदयात्रा', 'करेंगे', '।']
Number of tokens: 28 

WordPiece - Vocabulary Size: 50,000
Tokens: ['इसके', 'तहत', 'पार्टी', 'के', 'राज्य', 'से', 'लेकर', 'पंचायत', 'तक', 'के', 'पदाधिकारी', 'अपने

In [57]:
bpe_vocab_results = []

for vocab_size in [20_000, 30_000, 50_000]:

    stats = get_token_statistics(
        bpe_vocab_tokenizers[vocab_size],
        dev_df["sentence"].tolist()
    )

    stats["Tokenizer"] = "BPE"
    stats["Vocabulary Size"] = vocab_size

    bpe_vocab_results.append(stats)


bpe_vocab_results_df = pd.DataFrame(
    bpe_vocab_results
)

bpe_vocab_results_df

,Total Sentences,Total Tokens,Average Tokens,Minimum Tokens,Maximum Tokens,Tokenizer,Vocabulary Size
0,1000,24450,24.450,4,363,BPE,20000
1,1000,23928,23.928,4,359,BPE,30000
2,1000,23536,23.536,4,357,BPE,50000


In [58]:
wordpiece_vocab_results = []

for vocab_size in [20_000, 30_000, 50_000]:

    stats = get_token_statistics(
        wordpiece_vocab_tokenizers[vocab_size],
        dev_df["sentence"].tolist()
    )

    stats["Tokenizer"] = "WordPiece"
    stats["Vocabulary Size"] = vocab_size

    wordpiece_vocab_results.append(stats)


wordpiece_vocab_results_df = pd.DataFrame(
    wordpiece_vocab_results
)

wordpiece_vocab_results_df

,Total Sentences,Total Tokens,Average Tokens,Minimum Tokens,Maximum Tokens,Tokenizer,Vocabulary Size
0,1000,24745,24.745,4,367,WordPiece,20000
1,1000,24085,24.085,4,363,WordPiece,30000
2,1000,23640,23.640,4,360,WordPiece,50000


In [59]:
vocab_comparison = pd.concat(
    [
        bpe_vocab_results_df,
        wordpiece_vocab_results_df
    ],
    ignore_index=True
)

vocab_comparison

,Total Sentences,Total Tokens,Average Tokens,Minimum Tokens,Maximum Tokens,Tokenizer,Vocabulary Size
0,1000,24450,24.450,4,363,BPE,20000
1,1000,23928,23.928,4,359,BPE,30000
2,1000,23536,23.536,4,357,BPE,50000
3,1000,24745,24.745,4,367,WordPiece,20000
4,1000,24085,24.085,4,363,WordPiece,30000
5,1000,23640,23.640,4,360,WordPiece,50000


In [60]:
# --- Side-by-side pivot for Experiment 2: rows = Vocabulary Size, columns = Tokenizer ---
pivot_vocab_size = vocab_comparison.pivot(
    index="Vocabulary Size", columns="Tokenizer", values="Average Tokens"
)
pivot_vocab_size["BPE - WordPiece (diff)"] = (
    pivot_vocab_size["BPE"] - pivot_vocab_size["WordPiece"]
)
print("Average tokens per sentence, BPE vs WordPiece, by vocabulary size:")
pivot_vocab_size.round(3)

Average tokens per sentence, BPE vs WordPiece, by vocabulary size:


Tokenizer,BPE,WordPiece,BPE - WordPiece (diff)
Vocabulary Size,,,
20000,24.450,24.745,-0.295
30000,23.928,24.085,-0.157
50000,23.536,23.640,-0.104


## 8. Dev vs Test Evaluation (Generalization Check)

Re-running the same statistics on both the dev set and the untouched test set checks that the training-size trend from Experiment 1 isn't an artifact of one particular sample of sentences.


In [61]:
def evaluate_tokenizer(
    tokenizer,
    dev_sentences,
    test_sentences
):

    dev_stats = get_token_statistics(
        tokenizer,
        dev_sentences
    )

    test_stats = get_token_statistics(
        tokenizer,
        test_sentences
    )

    return dev_stats, test_stats

In [62]:
evaluation_results = []

for train_size in TRAIN_SIZES:

    tokenizer = bpe_tokenizers[train_size]

    dev_stats, test_stats = evaluate_tokenizer(
        tokenizer,
        dev_df["sentence"].tolist(),
        test_df["sentence"].tolist()
    )

    evaluation_results.append({
        "Tokenizer": "BPE",
        "Training Size": train_size,

        "Dev Avg Tokens":
            dev_stats["Average Tokens"],

        "Dev Total Tokens":
            dev_stats["Total Tokens"],

        "Test Avg Tokens":
            test_stats["Average Tokens"],

        "Test Total Tokens":
            test_stats["Total Tokens"]
    })

In [63]:
for train_size in TRAIN_SIZES:

    tokenizer = wordpiece_tokenizers[train_size]

    dev_stats, test_stats = evaluate_tokenizer(
        tokenizer,
        dev_df["sentence"].tolist(),
        test_df["sentence"].tolist()
    )

    evaluation_results.append({
        "Tokenizer": "WordPiece",
        "Training Size": train_size,

        "Dev Avg Tokens":
            dev_stats["Average Tokens"],

        "Dev Total Tokens":
            dev_stats["Total Tokens"],

        "Test Avg Tokens":
            test_stats["Average Tokens"],

        "Test Total Tokens":
            test_stats["Total Tokens"]
    })


evaluation_df = pd.DataFrame(
    evaluation_results
)

evaluation_df.round(2)

,Tokenizer,Training Size,Dev Avg Tokens,Dev Total Tokens,Test Avg Tokens,Test Total Tokens
0,BPE,100000,24.45,24450,24.69,24687
1,BPE,300000,24.40,24399,24.67,24672
2,BPE,500000,24.40,24405,24.67,24673
3,BPE,1000000,24.43,24428,24.69,24693
4,WordPiece,100000,24.74,24744,25.00,25003
5,WordPiece,300000,24.74,24739,24.99,24994
6,WordPiece,500000,24.76,24761,25.00,24999
7,WordPiece,1000000,24.79,24792,25.06,25060


In [65]:
training_size_comparison.round(2)

,Tokenizer,Training Size,Total Sentences,Total Tokens,Average Tokens,Minimum Tokens,Maximum Tokens
0,BPE,100000,1000,24450,24.45,4,363
1,BPE,300000,1000,24399,24.40,4,365
2,BPE,500000,1000,24405,24.40,4,365
3,BPE,1000000,1000,24428,24.43,4,366
4,WordPiece,100000,1000,24744,24.74,4,367
5,WordPiece,300000,1000,24739,24.74,4,371
6,WordPiece,500000,1000,24761,24.76,4,372
7,WordPiece,1000000,1000,24792,24.79,4,374


In [66]:
vocab_comparison.round(2)

,Total Sentences,Total Tokens,Average Tokens,Minimum Tokens,Maximum Tokens,Tokenizer,Vocabulary Size
0,1000,24450,24.45,4,363,BPE,20000
1,1000,23928,23.93,4,359,BPE,30000
2,1000,23536,23.54,4,357,BPE,50000
3,1000,24745,24.74,4,367,WordPiece,20000
4,1000,24085,24.08,4,363,WordPiece,30000
5,1000,23640,23.64,4,360,WordPiece,50000


In [67]:
training_size_comparison.to_csv(
    "training_size_comparison.csv",
    index=False
)

vocab_comparison.to_csv(
    "vocabulary_size_comparison.csv",
    index=False
)

evaluation_df.to_csv(
    "dev_test_evaluation.csv",
    index=False
)

bpe_vocab_results_df.to_csv(
    "bpe_vocabulary_results.csv",
    index=False
)

wordpiece_vocab_results_df.to_csv(
    "wordpiece_vocabulary_results.csv",
    index=False
)

print("All results saved successfully.")

All results saved successfully.


## 11. Summary — What to Say in the Viva

**Pipeline recap:** IndicCorpV2 (Hindi) → regex sentence splitting on `। ? !` → length-stratified train/dev/test split (so dev/test mirror the true sentence-length distribution) → nested training subsets (100K–1M) → BPE and WordPiece trained with the HuggingFace `tokenizers` library.

**Two controlled experiments, one variable changed at a time:**
1. **Training data size** (vocab fixed at 20,000) → tests whether *more data* helps the tokenizer learn better merges.
2. **Vocabulary size** (training data fixed at 100,000) → tests whether a *bigger vocabulary* lets the tokenizer represent text with fewer tokens.

**Metric:** average tokens per sentence on the held-out dev set (lower = more compression = fewer tokens the downstream language model has to process per sentence, which usually means cheaper training/inference and often better modeling of long-range context).

**How BPE and WordPiece differ (mechanism, not just results):**
- BPE merges the *most frequent* adjacent symbol pair.
- WordPiece merges the pair that maximizes a *likelihood score* `freq(pair) / (freq(left) * freq(right))`, which can favor rarer-but-more-informative merges over purely frequent ones.

**What the new tables/charts in this notebook let you point to directly:**
- The pivoted tables (`pivot_train_size`, `pivot_vocab_size`) put BPE and WordPiece as *side-by-side columns* with an explicit difference column, instead of stacked rows — much faster to read out loud.
- The grouped bar charts put BPE and WordPiece bars next to each other at every setting, so the difference is visually obvious (the line charts only show trend, not per-setting comparison).
- `compression_df` gives a plain-English number: *how many times more/fewer tokens than naive whitespace splitting* each configuration needs — a good one-line answer if asked "so what does this all mean practically?"
